# 01 — Country and Exchange-Rate Data

This notebook retrieves a country's currency and loads current and historical exchange rates.

Reusable API functions are maintained in `currency_explorer/data.py`.

## Plan

1. Configure the project path and API key.
2. Retrieve a country's primary currency.
3. Retrieve the latest exchange rate.
4. Load historical rates for forecasting.
5. Validate the resulting data.

## Setup and imports

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

In [ ]:
project_root = Path.cwd()

if not (project_root / "currency_explorer").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

load_dotenv(project_root / ".env", override=True)
api_key = os.getenv("API_KEY")

if not api_key:
    raise RuntimeError("API_KEY is not configured in .env")

print("Project root:", project_root)
print("API key loaded:", bool(api_key))

In [ ]:
from currency_explorer.data import (
    get_country_currency,
    get_exchange_rate,
    get_historical_rates,
)

## Country and current rate

In [ ]:
country = "Germany"
target_currency = "USD"

country_info = get_country_currency(country, api_key)
country_info

In [ ]:
current_rate = get_exchange_rate(
    base_currency=country_info["currency"],
    target_currency=target_currency,
)

current_result = {
    "country": country_info["country"],
    "base_currency": country_info["currency"],
    "target_currency": target_currency,
    "current_rate": current_rate,
}

pd.DataFrame([current_result])

## Historical rates

Frankfurter publishes observations for business days, so weekends and some holidays are absent.

In [ ]:
rates_df = get_historical_rates(
    base_currency=country_info["currency"],
    target_currency=target_currency,
    days=730,
)

print("Observations:", len(rates_df))
print("First date:", rates_df["date"].min())
print("Last date:", rates_df["date"].max())

rates_df.head()

## Data validation

In [ ]:
validation = {
    "missing_dates": int(rates_df["date"].isna().sum()),
    "missing_rates": int(rates_df["rate"].isna().sum()),
    "duplicate_dates": int(rates_df["date"].duplicated().sum()),
    "dates_are_sorted": bool(rates_df["date"].is_monotonic_increasing),
}

pd.Series(validation, name="value")

In [ ]:
rates_df["rate"].describe()